In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### Constants

In [2]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
str_dirname_output = './output'
str_variant = 'noPTImodel10'

Project: 20231010-gen-xii
Task: 08_retro_scoring


### Output directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Variant directory

In [4]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except:
    pass

### Get data

In [5]:
%%time

str_filename = 'df_retro_predictions.gzip'
str_uri = f's3://{str_project}/08_retro_scoring/07_analysis/{str_variant}/{str_filename}'
df = pd.read_parquet(str_uri)

# sort
df.sort_values(by=['bigAccountId','BITDEBTOR'], ascending=[True, False], inplace=True)

# show
df

CPU times: user 4.54 s, sys: 1.98 s, total: 6.52 s
Wall time: 2.26 s


,uniqueid__app_x,bigstatusid__app,strcity__app,strname__app,strzipcode__app,bitapproved__app,bitsystemdecline__app,bitfunded__app,applicationmonth__app,applicationquarter__app,...,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age,yhat_ad,yhat_pricing_pd,yhat_pricing_lgd
0,0__0__20210120,14.0,stanley,virginia,22851,0.0,0.0,0.0,1.0,1.0,...,1.080027,1,1,0.09,1.269231,5.0,9.567123,0.454956,0.457796,0.637971
5,0__0__20210120,14.0,chico,california,95926,0.0,0.0,0.0,1.0,1.0,...,1.080027,1,1,0.12,1.285714,6.0,7.263014,0.440640,0.118641,0.649063
1,0__0__20210120,14.0,austin,texas,78727,0.0,0.0,0.0,1.0,1.0,...,1.080027,1,1,0.12,1.153846,0.0,8.260274,0.533774,0.384922,0.643923
20,0__0__20210120,14.0,phoenix,arizona,85008,0.0,0.0,0.0,1.0,1.0,...,1.080027,1,1,0.06,1.578947,5.0,10.320548,0.485115,0.236593,0.640644
6,0__0__20210121,14.0,london,kentucky,40741,0.0,0.0,0.0,1.0,1.0,...,1.080027,1,1,0.15,1.375000,4.0,4.736986,0.459449,0.251424,0.621152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70312,0__0__20231127,1.0,joshua,texas,76058,0.0,0.0,0.0,11.0,4.0,...,0.970000,11,4,0.15,1.444444,6.0,1.912329,0.312679,0.384166,0.580512
70308,0__0__20231127,1.0,joshua,texas,76058,0.0,0.0,0.0,11.0,4.0,...,0.970000,11,4,0.12,1.444444,6.0,1.912329,0.579190,0.203354,0.674070
71120,0__0__20231127,1.0,florissant,missouri,63031,0.0,0.0,0.0,11.0,4.0,...,0.970000,11,4,0.09,1.181818,2.0,5.553425,0.155877,0.096083,0.647022
70995,0__0__20231127,1.0,sheboygan,wisconsin,53083,0.0,0.0,0.0,11.0,4.0,...,0.970000,11,4,0.06,1.156250,6.0,0.293151,0.237313,0.121283,0.669731


### Get grouped scores at the account level

In [6]:
df_tmp = df.groupby(by='bigAccountId', as_index=False).agg({
    'intopenbktype__app': 'first',
    'yhat_pricing_pd': 'mean',
    'yhat_pricing_lgd': 'mean',
})

# ecnl
str_colname = 'ECNL_production'
if str_variant == 'noPTImodel10':
    df_tmp[str_colname] = (df_tmp['yhat_pricing_pd'] * df_tmp['yhat_pricing_lgd']) * 2.36
else:
    df_tmp[str_colname] = (df_tmp['yhat_pricing_pd'] * df_tmp['yhat_pricing_lgd'])

# show
df_tmp

,bigAccountId,intopenbktype__app,yhat_pricing_pd,yhat_pricing_lgd,ECNL_production
0,5514485,nan,0.457796,0.637971,0.689263
1,5514970,nan,0.118641,0.649063,0.181733
2,5515245,nan,0.384922,0.643923,0.584950
3,5515340,nan,0.236593,0.640644,0.357709
4,5515580,nan,0.290466,0.629329,0.431404
...,...,...,...,...,...
58426,7373944,nan,0.177340,0.680696,0.284887
58427,7373949,nan,0.293760,0.627291,0.434884
58428,7373986,13.0,0.096083,0.647022,0.146717
58429,7374018,13.0,0.121283,0.669731,0.191695


### Write aggregated data to s3

In [7]:
%%time

str_filename = 'df.csv'
str_uri = f's3://{str_project}/08_retro_scoring/08_segment_distributions/02_gen_xii_production/{str_filename}'
df_tmp.to_csv(str_uri, index=False)

CPU times: user 389 ms, sys: 89.6 ms, total: 479 ms
Wall time: 567 ms
